# Laptop lab 4: Missing values, trees and forests
*Keep every listing, and let many trees decide*

**Machine Learning with Python** · Minor Amsterdam Data Science and Artificial Intelligence · Fall 2026<br>
Thursday, November 19, 2026 · Laptop labs: Batuhan Ataş (b.atas@uva.nl)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/batuatas/MLwPython-labs/blob/main/lab04.ipynb)

> **Before you start:** choose **File → Save a copy in Drive** and work in that copy. Colab does not save your changes in the original notebook.

This week you learned to fill the gaps in a table and to keep the features that matter (Lecture 7), and you met decision trees and random forests (Lecture 8). Today you use both on the Airbnb listings of Lab 3.

**The story.** The question is the same as in Lab 3: is a listing **premium**, more than €300 per night? But two things have changed. In Lab 3 we quietly dropped every listing with a missing value, about one in six. Today they stay: a pipeline that stops at a gap cannot help a host whose listing has no reviews yet. And every listing now comes with its **amenities**: Wifi, a dishwasher, a bathtub, 95 of them, each a 0/1 column. As before, 2,000 other listings stay locked away, gaps included: the **test set**.

**Your mission.** Build one pipeline that classifies *every* listing as premium or not, gaps and all, and estimate its accuracy on new listings. The warm-up shows you the pieces; then you decide how to put them together.

## How this lab works

| You see | What you do |
|---|---|
| **Run** | Run the cell and look at the output. |
| **Your turn** | Write the code yourself, or fill in the dots `...`. A check below the cell shows an orange *Not yet* box until you are done. |
| **Toolbox** | Click to unfold a piece of code that you can copy and change. |
| **Think** | Double-click *Your answer* and write a sentence or two. We discuss the answers together. |
| **Stop** | Wait here until the tutor says you can go on. |

Imputers and trees work like every other step in scikit-learn:

```python
imputer = SimpleImputer(strategy="median")    # fills every gap of a column with its median
imputer.fit(X_train)                          # learns one median per column
imputer.transform(X_new)                      # the gaps become those medians

tree = DecisionTreeClassifier(max_depth=3)    # a tree of at most 3 questions per path
tree.fit(X_train, y_train)
tree.predict(X_new)                           # the majority class of the leaf
```

| Part | Minutes |
|---|---|
| 0. Setup | 5 |
| 1–3. Warm-up: gaps, a tree, a forest | together on the screen, 15 |
| 4. Your mission | 45 |
| 5. The test set is unlocked | 5 |
| 6. What happened? | together, 15 |
| Check your understanding | together, at the end |

## Task 0. Setup

Run the cell below. It loads the libraries, downloads the tools of this lab and the data from Inside Airbnb, and cleans them as in Lab 1, but this time without dropping the listings with gaps. You should see a green box with a ✓.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# The tools of this lab are in the course lab repository;
# they also download the data from Inside Airbnb.
BASE = "https://raw.githubusercontent.com/batuatas/MLwPython-labs/main/"
import urllib.request
urllib.request.urlretrieve(BASE + "lab04_tools.py", "lab04_tools.py")
from lab04_tools import *

X, y = load_training_data()    # your 1,500 listings: features X, and y = 1 for premium
setup_ok()

**Run.** Your data: one row per listing, `y` = 1 if it costs more than €300 per night. `X` has three groups of columns, each in a list:

| List | Columns |
|---|---|
| `NUMERIC` | the 12 numbers of Lab 3: `accommodates`, `bedrooms`, `beds`, `bathrooms`, `minimum_nights`, `availability_365`, `number_of_reviews`, `reviews_per_month`, `review_scores_rating`, `host_listings`, `superhost`, `km_to_center` |
| `CATEGORICAL` | the 2 text columns of Lab 3: `room_type`, `neighbourhood` |
| `AMENITIES` | 95 new 0/1 columns, one per amenity that at least 5% of the listings offer: `Wifi`, `Dishwasher`, `Bathtub`, ... |

Some numbers are missing: in pandas, a gap is `NaN` ("not a number").

In [ ]:
print(X.shape, "  share of premium listings:", round(y.mean(), 2))
X.head()

## Warm-up: the pieces, one by one (Tasks 1 to 3)

We go through Tasks 1 to 3 together on the screen. Just run the cells and follow along; there is nothing to fill in. Everything you see here, you will reuse in your mission.

## Task 1. The gaps

**Run.** Where are the gaps, and how many listings have at least one? (Lecture 7, "The gaps in the survey".)

In [ ]:
gaps = X[NUMERIC].isna().sum()
print(gaps[gaps > 0])
print()
print("listings with at least one gap:", X[NUMERIC].isna().any(axis=1).sum(), "of", len(X))

**Run.** Why is a value missing? Before you fill anything in, look at who has the gap (Lecture 7, "Why is it missing?"). Here: which listings have no review score?

In [ ]:
no_score = X["review_scores_rating"].isna()
pd.crosstab(no_score, X["number_of_reviews"] == 0,
            rownames=["no review score"], colnames=["no reviews at all"])

Every listing without a review score has no reviews at all, and every listing without reviews has no score: the gap **depends on another column**. The same holds for `reviews_per_month`. That tells you how to fill it: a listing without reviews gets 0 reviews per month, not the median. For the score itself there is no right number: "no score yet" is something else than an average score.

**Run.** Is the gap itself informative? Compare the share of premium listings with and without a gap (Lecture 7, "The absence is informative").

In [ ]:
for col in ["minimum_nights", "beds", "review_scores_rating"]:
    gap = X[col].isna()
    print(f"{col:22} premium among listings with a gap: {y[gap].mean():.2f}   without: {y[~gap].mean():.2f}")

Listings without a minimum stay are rarely premium. We do not know why (they are all private rooms, so perhaps rooms in bed and breakfasts), but the gap carries information. `SimpleImputer(add_indicator=True)` keeps it: next to the filled column, it adds a 0/1 column "was missing".

**Run.** What if we do nothing? Logistic regression stops at the first `NaN`. Dropping the listings with gaps (as in Lab 3) works for training, but then the model cannot say anything about a new listing with a gap, and the locked test set has them too.

In [ ]:
try:
    cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), X[NUMERIC], y, cv=5,
                    error_score="raise")
except ValueError as e:
    print("Error:", str(e).split("\n")[0][:90])

print("listings without a gap:", len(X.dropna(subset=NUMERIC)), "of", len(X))

**Run.** The imputer goes **inside the pipeline**, before the scaler, so that it learns its medians from the training part of every fold (Lecture 7, "The imputer belongs in the pipeline"). Three ways to fill the gaps, each in the pipeline of Lab 3:

In [ ]:
fills = {
    "median": SimpleImputer(strategy="median"),
    "median + indicator": SimpleImputer(strategy="median", add_indicator=True),
    "5 nearest neighbors": make_pipeline(StandardScaler(), KNNImputer(n_neighbors=5)),  # distances: scale first
}
for name, imputer in fills.items():
    pipe = make_pipeline(
        make_column_transformer(
            (make_pipeline(imputer, StandardScaler()), NUMERIC),     # numbers: fill, then scale
            (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)),  # text: one-hot
        LogisticRegression(C=0.1))
    print(f"{name:20} mean validation accuracy {cross_val_score(pipe, X, y, cv=5).mean():.3f}")

With one listing in six having a gap, the choice hardly matters, just as in Lecture 7. What matters is that every listing now gets a prediction.

## Task 2. A decision tree

**Run.** A tree of depth 2 on two features: the number of guests and the distance to the center. Read it from the top: every box asks one yes/no question; a listing goes left if the answer is yes (Lecture 8, "A tree asks yes/no questions").

In [ ]:
two = ["accommodates", "km_to_center"]
tree = DecisionTreeClassifier(max_depth=2, random_state=0)
tree.fit(X[two], y)

fig, ax = plt.subplots(figsize=(13, 5))
plot_tree(tree, feature_names=two, class_names=["not premium", "premium"], filled=True, ax=ax)
plt.show()

In every box: the question, the Gini impurity (0 for a pure group, 0.5 for a fifty-fifty group), the number of listings (`samples`), how many of each class (`value`), and the majority class, which is what the tree predicts there.

**Run.** The same idea, drawn on the data with the plotting code of Lab 3: a shallow tree, a deeper one, and a full tree. Every question is a cut parallel to an axis, so the regions are rectangles (Lecture 8, "Deeper trees, finer rectangles").

In [ ]:
guests = np.linspace(0.5, 10.5, 200)
km = np.linspace(0, 8, 200)
G, K = np.meshgrid(guests, km)
grid = pd.DataFrame({"accommodates": G.ravel(), "km_to_center": K.ravel()})
jitter = np.random.default_rng(0).uniform(-0.3, 0.3, len(X))     # guests are whole numbers
premium = (y == 1).to_numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, depth in zip(axes, [2, 5, None]):
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X[two], y)
    P = t.predict_proba(grid)[:, 1].reshape(G.shape)
    ax.contourf(G, K, P, levels=np.linspace(0, 1, 11), cmap="RdBu", alpha=0.6)
    ax.scatter(X["accommodates"][~premium] + jitter[~premium], X["km_to_center"][~premium],
               marker="o", s=8, color="#C0002A", label="not premium")
    ax.scatter(X["accommodates"][premium] + jitter[premium], X["km_to_center"][premium],
               marker="^", s=8, color="#25567B", label="premium")
    ax.set_title(f"depth {depth or 'full'}: training accuracy {t.score(X[two], y):.2f}")
    ax.set_xlabel("guests")
    ax.set_xlim(0.5, 10.5)
    ax.set_ylim(0, 8)
axes[0].set_ylabel("distance to the center (km)")
axes[0].legend(loc="upper right")
plt.show()

**Run.** How deep should the tree be? Now on all features. A tree needs **no scaler**, since a threshold does not care about units, but it still needs numbers, so the text columns are one-hot encoded, and the gaps are filled. The grid search tries depths 1 to 15; `return_train_score=True` also keeps the training accuracy (Lecture 8, "Choosing the depth").

In [ ]:
prep_tree = make_column_transformer(
    (SimpleImputer(strategy="median", add_indicator=True), NUMERIC),   # fill; no scaler for a tree
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
tree_pipe = make_pipeline(prep_tree, DecisionTreeClassifier(random_state=0))

depths = list(range(1, 16))
grid = GridSearchCV(tree_pipe, {"decisiontreeclassifier__max_depth": depths}, cv=5, return_train_score=True)
grid.fit(X, y)
print("best depth:", grid.best_params_, "  mean validation accuracy:", round(grid.best_score_, 3))

fig, ax = plt.subplots()
ax.plot(depths, grid.cv_results_["mean_train_score"], marker="o", label="training")
ax.plot(depths, grid.cv_results_["mean_test_score"], marker="o", label="validation (5-fold)")
ax.set_xlabel("max_depth")
ax.set_ylabel("accuracy")
ax.legend()
plt.show()

The picture of Lecture 3 again: the training accuracy keeps climbing towards 1, the validation accuracy stays flat and then drops. A deep tree memorizes the training listings.

## Task 3. A random forest, and what it finds important

**Run.** A forest of 200 trees, each grown fully on its own bootstrap sample, each question choosing among a random subset of the features; then they vote (Lecture 8, "Random forest"). This time with the amenities: a third group of columns that passes through unchanged (`"passthrough"`). `oob_score=True` tests every tree on the listings its bootstrap sample left out: a validation score for free.

In [ ]:
prep_forest = make_column_transformer(
    (SimpleImputer(strategy="median", add_indicator=True), NUMERIC),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ("passthrough", AMENITIES),                    # the 95 amenities: already 0/1, keep as they are
    verbose_feature_names_out=False)               # keep the plain column names
forest = make_pipeline(prep_forest, RandomForestClassifier(n_estimators=200, oob_score=True,
                                                           random_state=0, n_jobs=-1))

print("mean validation accuracy:", round(cross_val_score(forest, X, y, cv=5).mean(), 3))
forest.fit(X, y)
print("out-of-bag accuracy:     ", round(forest[-1].oob_score_, 3))

**Run.** Which features does the forest use most? `feature_importances_` says how much each feature reduced the impurity, summed over all questions of all trees (Lecture 8, "Feature importances").

In [ ]:
importances = pd.Series(forest[-1].feature_importances_, index=forest[0].get_feature_names_out())
fig, ax = plt.subplots(figsize=(7, 5))
importances.sort_values().tail(15).plot.barh(ax=ax)
ax.set_xlabel("importance in the forest")
plt.show()

Read this **with doubt** (Lecture 8, "Read with doubt"). An importance has no sign: it does not say whether a dishwasher makes a listing more or less likely to be premium. It says nothing about causes: buying a dishwasher will not make a room premium. And twins share: guests, bedrooms and beds say almost the same thing, so they split the credit between them.

## Task 4. Your mission

Now it is your turn. Build **one pipeline** that classifies every listing as premium or not, **gaps included**, and estimate its accuracy on the locked listings.

You do this in five short steps. Step 1 is ready to run. In Steps 2 to 5 you write the code yourself: the comments in each cell say what to do, and you can copy from Tasks 1 to 3, from Step 1 and from the **toolbox** right below. After every run, ask yourself: did the mean validation accuracy go up or down? Under every step you find where to look if you are stuck.

**The rule:** you may do anything with `X` and `y`. The test set stays locked. Your pipeline must work for listings with gaps.

Write down your results in the notes cell at the end of the mission; you will need them to choose.

<details><summary><b>Toolbox</b> (click to unfold)</summary>

### 1. Filling the gaps inside the pipeline

An imputer is one more step, before the scaler. In the column transformer, give the numbers a small pipeline: first fill, then scale.

```python
(make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC)
```

The choices (Lecture 7, "Fill in a constant" to "Neighbors and models"):

```python
SimpleImputer(strategy="median")                       # the median of the column: the default
SimpleImputer(strategy="median", add_indicator=True)   # also a 0/1 column "was missing" per column with gaps
SimpleImputer(strategy="constant", fill_value=0)       # a number you choose, from knowledge of the problem
make_pipeline(StandardScaler(), KNNImputer(n_neighbors=5))   # the average of the 5 most similar listings
```

`KNNImputer` measures distances, so scale before it (`StandardScaler` simply skips the gaps). Different columns can get different fills: give each group its own line, as in Lab 3.

```python
zero_fill = ["reviews_per_month"]                       # no reviews: 0 per month
other = [c for c in NUMERIC if c not in zero_fill]

prep = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="constant", fill_value=0), StandardScaler()), zero_fill),
    (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), other),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
```

### 2. A decision tree, and its size

A tree needs no scaler, but it needs numbers and (in this lab) filled gaps:

```python
prep_tree = make_column_transformer(
    (SimpleImputer(strategy="median", add_indicator=True), NUMERIC),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))

my_tree = make_pipeline(prep_tree, DecisionTreeClassifier(max_depth=4, random_state=0))
```

The size of the tree is its complexity, like `k` in $k$-NN and `C` in logistic regression. One parameter is enough; choose it with a grid search (Lecture 8, "Keep the tree small"):

```python
grid = GridSearchCV(my_tree, {"decisiontreeclassifier__max_depth": range(1, 16)}, cv=5)
# or: {"decisiontreeclassifier__min_samples_leaf": [1, 5, 10, 20, 50]}
```

### 3. A random forest

Change only the model: the trees vote. More trees never hurt, they only cost time; `max_features` (how many features each question may choose from) is the one setting worth tuning (Lecture 8, "How many trees? How many features per question?").

```python
my_forest = make_pipeline(prep_tree, RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1))

grid = GridSearchCV(my_forest, {"randomforestclassifier__max_features": ["sqrt", 0.2, 0.4, None]}, cv=5)
```

`"sqrt"` (the default) lets each question choose among $\sqrt{p}$ of the $p$ features; `0.2` among 20% of them; `None` among all of them, which makes every tree almost the same. `n_jobs=-1` uses all processors of the computer.

### 4. The amenities, and choosing among them

The amenities are already 0/1: add them as a third group that passes through unchanged.

```python
prep_all = make_column_transformer(
    (SimpleImputer(strategy="median", add_indicator=True), NUMERIC),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ("passthrough", AMENITIES))
```

A forest chooses among them by itself. A linear model may do better with fewer: put a selection step **after the column transformer and before the model**, inside the pipeline, so that it is learned again in every fold (Lecture 7, "The leak"). The number of features `k` is a setting: let a grid search choose it.

```python
prep_lin = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), NUMERIC),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ("passthrough", AMENITIES))

my_select = make_pipeline(prep_lin, SelectKBest(f_classif, k=20), LogisticRegression(C=0.1, max_iter=1000))
grid = GridSearchCV(my_select, {"selectkbest__k": [5, 10, 20, 40, 80, "all"]}, cv=5)
```

Or let a model choose (Lecture 7, "Ask a model"): `SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=0))` keeps the features with an importance above the average.

### 5. Extra: gradient boosting

Many small trees in sequence, each fitted to the errors of the trees before it (Lecture 8, "Gradient boosting"). `HistGradientBoostingClassifier` even handles the gaps by itself, so it needs no imputer; it needs a dense table, hence `sparse_threshold=0`.

```python
prep_boost = make_column_transformer(
    ("passthrough", NUMERIC),                               # gaps and all
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ("passthrough", AMENITIES),
    sparse_threshold=0)
my_boost = make_pipeline(prep_boost, HistGradientBoostingClassifier(random_state=0))
```

Its main settings: `learning_rate` (0.1 by default) and `max_iter`, the number of trees (100).

### Names in a grid search

A setting inside a pipeline is called by the name of its step in lower case, two underscores, and its own name: `logisticregression__C`, `decisiontreeclassifier__max_depth`, `randomforestclassifier__max_features`, `selectkbest__k`, `histgradientboostingclassifier__learning_rate`. A grid search can try several settings at once: `{"selectkbest__k": [10, 40], "logisticregression__C": [0.01, 0.1, 1]}` tries all six combinations.

</details>

### Step 1. The starting pipeline

**Run.** The pipeline of Lab 3, with one new step: a median imputer before the scaler. It is your starting point.

In [ ]:
# Step 1. Run this cell. Nothing to change.
my_pipe = make_pipeline(
    make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
        (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)),
    LogisticRegression(C=0.1))

scores = cross_val_score(my_pipe, X, y, cv=5)
print("Step 1, starting pipeline: mean validation accuracy", round(scores.mean(), 3))

*Stuck?* Toolbox, part 1 · Cheat sheet, section 2 "Filling the gaps" · Lecture 7, "The imputer belongs in the pipeline".

### Step 2. Fill the gaps your way

Task 1 showed that the gaps have reasons. Use them: a different fill for a column whose gap you understand, an indicator for a gap that carries information, or neighbors.

In [ ]:
# Step 2. Fill the gaps your way (Toolbox, part 1).
# 1. Copy the pipeline of Step 1, with its two lines for the score.
# 2. Change the imputer: add_indicator=True, KNNImputer, or a separate group for
#    reviews_per_month filled with 0.
# 3. Run, and compare the mean validation accuracy with Step 1.

*Stuck?* Toolbox, part 1 · Cheat sheet, section 2 "Filling the gaps" · Lecture 7, "Fill in a constant", "The absence is informative" and "Neighbors and models".

### Step 3. A decision tree

Replace logistic regression by a decision tree, and let a grid search choose its depth. How does it compare with Step 2?

In [ ]:
# Step 3. A decision tree (Toolbox, part 2).
# 1. Make a pipeline: prep_tree of Task 2 (imputer and one-hot, no scaler), then
#    DecisionTreeClassifier(random_state=0).
# 2. Make a GridSearchCV with {"decisiontreeclassifier__max_depth": range(1, 16)} and cv=5,
#    and fit it on X and y.
# 3. Print grid.best_params_ and grid.best_score_.
# Extra: try min_samples_leaf instead of max_depth.

*Stuck?* Toolbox, part 2 · Cheat sheet, section 4 "Decision trees" · Lecture 8, "Trees in scikit-learn" and "Choosing the depth".

### Step 4. A random forest

Many trees instead of one. Does the forest beat the single tree? Does it beat logistic regression?

In [ ]:
# Step 4. A random forest (Toolbox, part 3).
# 1. Copy your tree pipeline of Step 3, and replace the tree by
#    RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).
# 2. Run, and compare with Steps 2 and 3.
# 3. Choose max_features with a grid search: {"randomforestclassifier__max_features": ["sqrt", 0.2, 0.4, None]}.

*Stuck?* Toolbox, part 3 · Cheat sheet, section 5 "Random forests" · Lecture 8, "Random forest" and "How many trees? How many features per question?".

### Step 5. The amenities

Do the 95 amenities help? Add them to your best pipeline so far. For a linear model, also try to keep only the best ones with a selection step.

In [ ]:
# Step 5. The amenities (Toolbox, part 4).
# 1. Add ("passthrough", AMENITIES) as a third group to the column transformer of your forest
#    (or of your logistic regression), run, and compare.
# 2. For logistic regression: add SelectKBest(f_classif, k=...) between the column transformer and
#    the model, and let a grid search choose k: {"selectkbest__k": [5, 10, 20, 40, 80, "all"]}.
# Extra: gradient boosting (Toolbox, part 5).

*Stuck?* Toolbox, parts 4 and 5 · Cheat sheet, section 3 "Choosing features" · Lecture 7, "Score each feature alone", "How many to keep?" and "The honest version is shorter".

### Your notes

**Your turn.** Double-click the cell below and write down, for each step, what you tried and its mean validation accuracy. Then choose your best pipeline.

*Your notes:*

- Step 1, starting pipeline: ...
- Step 2, my fill: ...
- Step 3, a tree: ...
- Step 4, a forest: ...
- Step 5, the amenities: ...
- My choice: ...

### Your submission

**Your turn.** Fill in your pipeline, a few words on how you chose it, and the accuracy you expect on the locked listings (your best mean validation accuracy is a good estimate). The last line fits your pipeline once more on **all** 1,500 training listings.

In [ ]:
my_pipeline = ...      # your best pipeline, for example grid_forest.best_estimator_ (Step 4)
my_strategy = "..."    # how you chose it, in your own words
my_estimate = ...      # the accuracy you expect on new listings, for example 0.75

my_pipeline.fit(X, y)  # learn from all 1,500 training listings

In [ ]:
# Check
check_submission()

> **Stop.** Wait here until the tutor unlocks the test set. Meanwhile, compare your pipeline with your neighbor's: the same model? The same fill?

## Task 5. The test set is unlocked

**Your turn.** When the tutor shows the password on the screen, type it in quotes and run the cell. Then run the next cell: it scores your pipeline on the 2,000 listings that no model has seen, gaps included.

In [ ]:
X_test, y_test = unlock_test_set("...")    # the password on the screen

In [ ]:
test_accuracy = report(my_pipeline, my_estimate, X_test, y_test)

Tell the tutor your pipeline, how you chose it, your estimate and your test accuracy. We collect them on the screen.

> **As in Labs 2 and 3:** the test set has now been used. Changing your pipeline to do better on it would make its score an optimistic estimate.

## Task 6. What happened?

**Run.** The pieces of today, one by one: for each pipeline, the mean 5-fold cross-validation accuracy on your 1,500 listings, and the accuracy on the test set. This takes about a minute.

In [ ]:
check_unlocked(X_test)      # stops here while the test set is still locked
from sklearn.dummy import DummyClassifier
lin = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
lin_ind = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), NUMERIC),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
trees = make_column_transformer(
    (SimpleImputer(strategy="median", add_indicator=True), NUMERIC),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
trees_all = make_column_transformer(
    (SimpleImputer(strategy="median", add_indicator=True), NUMERIC),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ("passthrough", AMENITIES))
candidates = {
    "always the most common class": DummyClassifier(),
    "logistic regression, median": make_pipeline(lin, LogisticRegression(C=0.1)),
    "  + missing indicators": make_pipeline(lin_ind, LogisticRegression(C=0.1)),
    "tree, depth 4": make_pipeline(trees, DecisionTreeClassifier(max_depth=4, random_state=0)),
    "tree, full depth": make_pipeline(trees, DecisionTreeClassifier(random_state=0)),
    "forest, 200 trees": make_pipeline(trees, RandomForestClassifier(200, random_state=0, n_jobs=-1)),
    "  + amenities": make_pipeline(trees_all, RandomForestClassifier(200, random_state=0, n_jobs=-1)),
}
rows = []
for name, m in candidates.items():
    m.fit(X, y)
    rows.append({"pipeline": name,
                 "training": m.score(X, y),
                 "5-fold CV": cross_val_score(m, X, y, cv=5).mean(),
                 "test set": m.score(X_test, y_test)})
pd.DataFrame(rows).set_index("pipeline").round(3)

**Think.** We discuss these together.

(a) The full tree: compare its training accuracy with its test accuracy. What is going on, and which setting prevents it? (Lecture 8, "Keep the tree small")<br>
(b) The forest grows full trees too. Why does it not have the same problem? (Lecture 8, "Random forest")<br>
(c) Forest or logistic regression: which would you choose here, and why? (Lecture 8, "Which model when?")<br>
(d) In Task 3, `Dishwasher` was among the most important features of the forest. Should a host buy a dishwasher to make the listing premium?<br>
(e) In Lab 3 we dropped the listings with gaps. What would that pipeline have done with the test listings that have a gap?

*Your answer:*

## Check your understanding

Answer each question for yourself first, then unfold the answer. We go through them together at the end of the lab.

**1.** A listing has no review score because it has no reviews yet. Which kind of gap is this?

A. By accident &nbsp; B. It depends on another column &nbsp; C. It depends on the missing value itself &nbsp; D. It is not a gap

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** Whether the score is missing follows from another column, `number_of_reviews` (Lecture 7, "Why is it missing?"). That column also tells you what the gap means.

</details>

**2.** A colleague chooses the 20 best amenities with `SelectKBest` on all 1,500 listings, and then reports the 5-fold cross-validation accuracy of a model on those 20. The estimate is

A. unbiased &nbsp; B. too optimistic &nbsp; C. too pessimistic

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** The selection looked at the labels of the validation folds too. Selection learns from the labels, so it belongs inside the pipeline (Lecture 7, "The leak").

</details>

**3.** A leaf of a tree holds 40 training listings, 30 of them premium. What does the tree predict for a new listing that lands there?

A. premium, with probability 0.75 &nbsp; B. not premium &nbsp; C. premium, with probability 1 &nbsp; D. It depends on the depth

<details><summary><b>Answer</b> (click to unfold)</summary>

**A.** The majority class of the leaf, with the share of that class as its probability: `predict_proba` gives [0.25, 0.75] (Lecture 8, "Trees in scikit-learn"). Its Gini impurity is 1 − 0.75² − 0.25² = 0.375.

</details>

**4.** You measure the distance to the center in meters instead of kilometers. What happens to a decision tree?

A. It gets better &nbsp; B. It gets worse &nbsp; C. The same tree, with thresholds 1,000 times larger &nbsp; D. It needs a scaler now

<details><summary><b>Answer</b> (click to unfold)</summary>

**C.** A question compares one feature with a threshold; changing the unit only changes the threshold. Trees need no scaling (Lecture 8, "Trees in scikit-learn").

</details>

**5.** You double the number of trees in a random forest, from 200 to 400. What happens?

A. It overfits more &nbsp; B. It costs more time, and the accuracy stays about the same or improves a little &nbsp; C. It needs a smaller `max_depth` &nbsp; D. It becomes a gradient boosting model

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** More trees never hurt, they only cost time (Lecture 8, "How many trees? How many features per question?").

</details>

## If you finish early

**A. The leak, by hand.** Add 1,000 columns of pure noise to the 12 numbers, and give every listing a random label. There is nothing to learn. Choose the 10 best columns with `SelectKBest` on all listings first, then cross-validate a logistic regression on them; then do the same with the selection inside the pipeline. Compare the two accuracies with 0.5 (Lecture 7, "The leak").

In [ ]:
...

**B. A tree cannot extrapolate.** Fit `DecisionTreeRegressor(max_depth=3)` on the number of guests alone, with `y` as a number: in every leaf it predicts the share of premium listings. Predict for 1, 4, 8, 16 and 32 guests. What happens beyond the largest listing in the training data? (Lecture 8, "Trees for regression".)

In [ ]:
...

## Where this is going

> Every step that learns from data, the imputer and the selection included, goes inside the pipeline. Then every listing gets a prediction, and cross-validation stays honest.

Today the gaps had reasons, and a forest of trees matched the linear model of Lab 3 without any scaling or tuning of features. No model wins everywhere: start simple, and let cross-validation decide.

**Coming up.**

- Lecture 9 (Monday, November 23): model evaluation. Accuracy has been our only yardstick so far; it is not always the right one.
- Lecture 10 (Tuesday, November 24): imbalanced data.

**Reading.** Müller and Guido (2016), Chapter 2: "Decision Trees" and "Ensembles of Decision Trees"; Chapter 4: "Automatic Feature Selection". Imputation: scikit-learn User Guide, "Imputation of missing values".

---
*This lab practices Lectures 7 and 8 of Ş. İlker Birbil. Data: [Inside Airbnb](https://insideairbnb.com), Amsterdam, collected on September 16, 2026, licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); downloaded directly from their website and cleaned as in Lab 1, without dropping the listings with gaps.*